# Layer Normalization (层归一化)

层归一化是Transformer中的关键组件，用于稳定训练和加速收敛。

## 核心概念

**公式：**
$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

其中：
- $\mu$ 是该层所有特征的均值
- $\sigma^2$ 是该层所有特征的方差
- $\gamma$ 和 $\beta$ 是可学习参数
- $\epsilon$ 是防止除零的小常数

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 1. Layer Normalization 实现

In [ ]:
class LayerNorm:
    """层归一化实现"""
    
    def __init__(self, normalized_shape, eps=1e-5, use_bias=True):
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.use_bias = use_bias
        
        # 可学习参数
        self.gamma = np.ones(normalized_shape)  # 缩放
        self.beta = np.zeros(normalized_shape) if use_bias else None  # 平移
    
    def forward(self, x):
        """前向传播"""
        # 计算均值和方差（在特征维度上）
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        
        # 标准化
        x_normalized = (x - mean) / np.sqrt(var + self.eps)
        
        # 缩放和平移
        out = self.gamma * x_normalized
        if self.use_bias:
            out = out + self.beta
        
        return out

# 测试
batch_size, seq_len, embed_dim = 2, 4, 6
x = np.random.randn(batch_size, seq_len, embed_dim) * 2 + 1

ln = LayerNorm(embed_dim)
output = ln.forward(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"\n第一个token归一化前: {x[0, 0]}")
print(f"第一个token归一化后: {output[0, 0]}")
print(f"\n归一化后的均值: {np.mean(output[0, 0]):.6f}")
print(f"归一化后的方差: {np.var(output[0, 0]):.6f}")

## 2. RMS Normalization 实现

RMSNorm是LayerNorm的简化版本，去掉了均值中心化，只使用RMS归一化。

**公式：**
$$\text{RMSNorm}(x) = \gamma \cdot \frac{x}{\sqrt{\text{mean}(x^2) + \epsilon}}$$

In [ ]:
class RMSNorm:
    """RMS归一化实现"""
    
    def __init__(self, normalized_shape, eps=1e-5):
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.gamma = np.ones(normalized_shape)
    
    def forward(self, x):
        """前向传播"""
        # 计算RMS（均方根）
        rms = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + self.eps)
        
        # 归一化并缩放
        out = self.gamma * (x / rms)
        
        return out

# 测试
rms = RMSNorm(embed_dim)
rms_output = rms.forward(x)

print(f"RMSNorm输出形状: {rms_output.shape}")
print(f"\nRMSNorm vs LayerNorm 差异:")
print(f"  平均绝对差异: {np.mean(np.abs(output - rms_output)):.6f}")
print(f"  最大绝对差异: {np.max(np.abs(output - rms_output)):.6f}")

## 3. Batch Normalization 实现（对比）

In [ ]:
class BatchNorm:
    """批归一化实现"""
    
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        self.gamma = np.ones(num_features)
        self.beta = np.zeros(num_features)
        
        # 移动平均（用于推理）
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)
        
        self.training = True
    
    def forward(self, x):
        """前向传播"""
        if self.training:
            # 在batch维度上计算统计量
            axes = tuple(range(x.ndim - 1))
            mean = np.mean(x, axis=axes)
            var = np.var(x, axis=axes)
            
            # 更新移动平均
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:
            mean = self.running_mean
            var = self.running_var
        
        # 标准化
        x_normalized = (x - mean) / np.sqrt(var + self.eps)
        
        # 缩放和平移
        out = self.gamma * x_normalized + self.beta
        
        return out

# 测试
bn = BatchNorm(embed_dim)
bn_output = bn.forward(x)

print(f"BatchNorm输出形状: {bn_output.shape}")
print(f"\nrunning_mean: {bn.running_mean}")
print(f"running_var: {bn.running_var}")

## 4. 可视化对比：Layer Norm vs Batch Norm

In [ ]:
# 创建示例数据
batch_size, features = 4, 8
x_2d = np.random.randn(batch_size, features) * 2 + np.arange(features)

# 应用不同的归一化
ln_2d = LayerNorm(features)
bn_2d = BatchNorm(features)

ln_out_2d = ln_2d.forward(x_2d)
bn_out_2d = bn_2d.forward(x_2d)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 原始数据
sns.heatmap(x_2d, annot=True, fmt=".2f", cmap="RdYlBu_r", ax=axes[0], cbar=True)
axes[0].set_title("Original Data")
axes[0].set_xlabel("Features")
axes[0].set_ylabel("Batch")

# Layer Norm（行归一化）
sns.heatmap(ln_out_2d, annot=True, fmt=".2f", cmap="RdYlBu_r", ax=axes[1], cbar=True)
axes[1].set_title("Layer Norm (行归一化)")
axes[1].set_xlabel("Features")
axes[1].set_ylabel("Batch")

# Batch Norm（列归一化）
sns.heatmap(bn_out_2d, annot=True, fmt=".2f", cmap="RdYlBu_r", ax=axes[2], cbar=True)
axes[2].set_title("Batch Norm (列归一化)")
axes[2].set_xlabel("Features")
axes[2].set_ylabel("Batch")

plt.tight_layout()
plt.show()

print("Layer Norm: 每行的均值≈0，方差≈1")
print("Batch Norm: 每列的均值≈0，方差≈1")

## 5. 归一化维度对比

In [ ]:
# 验证归一化效果
print("Layer Norm - 每个样本的特征统计:")
for i in range(batch_size):
    print(f"  样本{i}: 均值={np.mean(ln_out_2d[i]):.6f}, 方差={np.var(ln_out_2d[i]):.6f}")

print("\nBatch Norm - 每个特征的批统计:")
for j in range(min(4, features)):
    print(f"  特征{j}: 均值={np.mean(bn_out_2d[:, j]):.6f}, 方差={np.var(bn_out_2d[:, j]):.6f}")

## 6. 计算效率对比

In [ ]:
import time

# 创建大规模输入
large_x = np.random.randn(32, 512, 512)

# Layer Norm
ln_large = LayerNorm(512)
start = time.time()
for _ in range(100):
    _ = ln_large.forward(large_x)
ln_time = (time.time() - start) / 100

# RMS Norm
rms_large = RMSNorm(512)
start = time.time()
for _ in range(100):
    _ = rms_large.forward(large_x)
rms_time = (time.time() - start) / 100

# Batch Norm
bn_large = BatchNorm(512)
start = time.time()
for _ in range(100):
    _ = bn_large.forward(large_x)
bn_time = (time.time() - start) / 100

# 可视化
times = [ln_time * 1000, rms_time * 1000, bn_time * 1000]
labels = ['Layer Norm', 'RMS Norm', 'Batch Norm']

plt.figure(figsize=(10, 6))
bars = plt.bar(labels, times, color=['#3498db', '#2ecc71', '#e74c3c'])
plt.ylabel('Time (ms)')
plt.title('Normalization Methods - Computation Time Comparison')
plt.grid(axis='y', alpha=0.3)

# 添加数值标签
for bar, time_val in zip(bars, times):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{time_val:.2f}ms',
            ha='center', va='bottom')

plt.show()

print(f"输入形状: {large_x.shape}")
print(f"Layer Norm: {ln_time*1000:.2f}ms")
print(f"RMS Norm: {rms_time*1000:.2f}ms (加速 {ln_time/rms_time:.2f}x)")
print(f"Batch Norm: {bn_time*1000:.2f}ms")

## 7. Layer Norm vs RMS Norm 输出分布

In [ ]:
# 生成测试数据
test_x = np.random.randn(1000, 512)

ln_test = LayerNorm(512)
rms_test = RMSNorm(512)

ln_out_test = ln_test.forward(test_x)
rms_out_test = rms_test.forward(test_x)

# 绘制分布
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(test_x.flatten(), bins=50, alpha=0.7, color='gray')
axes[0].set_title('Original Distribution')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

axes[1].hist(ln_out_test.flatten(), bins=50, alpha=0.7, color='blue')
axes[1].set_title('Layer Norm Distribution')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')

axes[2].hist(rms_out_test.flatten(), bins=50, alpha=0.7, color='green')
axes[2].set_title('RMS Norm Distribution')
axes[2].set_xlabel('Value')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print("统计量对比:")
print(f"原始: 均值={np.mean(test_x):.4f}, 标准差={np.std(test_x):.4f}")
print(f"Layer Norm: 均值={np.mean(ln_out_test):.4f}, 标准差={np.std(ln_out_test):.4f}")
print(f"RMS Norm: 均值={np.mean(rms_out_test):.4f}, 标准差={np.std(rms_out_test):.4f}")

## 8. 实际应用：Transformer中的位置

### Post-LN (BERT风格)
```
x = x + Attention(x)
x = LayerNorm(x)
x = x + FFN(x)
x = LayerNorm(x)
```

### Pre-LN (GPT风格)
```
x = x + Attention(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

In [ ]:
# 模拟Transformer Block
def transformer_block_post_ln(x, attention_fn, ffn_fn, ln1, ln2):
    """Post-LN风格"""
    # Attention + Residual + LN
    x = ln1.forward(x + attention_fn(x))
    # FFN + Residual + LN
    x = ln2.forward(x + ffn_fn(x))
    return x

def transformer_block_pre_ln(x, attention_fn, ffn_fn, ln1, ln2):
    """Pre-LN风格"""
    # LN + Attention + Residual
    x = x + attention_fn(ln1.forward(x))
    # LN + FFN + Residual
    x = x + ffn_fn(ln2.forward(x))
    return x

# 示例函数
def dummy_attention(x):
    return x * 0.9

def dummy_ffn(x):
    return x * 1.1

# 测试
test_input = np.random.randn(4, 8, 64)
ln1, ln2 = LayerNorm(64), LayerNorm(64)

post_ln_out = transformer_block_post_ln(test_input.copy(), dummy_attention, dummy_ffn, ln1, ln2)
pre_ln_out = transformer_block_pre_ln(test_input.copy(), dummy_attention, dummy_ffn, 
                                       LayerNorm(64), LayerNorm(64))

print("Post-LN vs Pre-LN:")
print(f"Post-LN输出范围: [{np.min(post_ln_out):.4f}, {np.max(post_ln_out):.4f}]")
print(f"Pre-LN输出范围: [{np.min(pre_ln_out):.4f}, {np.max(pre_ln_out):.4f}]")
print("\nPre-LN通常训练更稳定，尤其是深层网络")

## 9. 总结

### Layer Norm 优势
- ✓ 不依赖batch大小，训练和推理一致
- ✓ 适合序列模型和小batch场景
- ✓ 在NLP任务中表现优异

### RMS Norm 优势
- ✓ 计算更简单，速度更快
- ✓ 参数更少（无beta）
- ✓ 在大模型中效果好（LLaMA等）

### Batch Norm vs Layer Norm
- Batch Norm：适合CV，依赖batch统计
- Layer Norm：适合NLP，每个样本独立

### 实际应用
- **BERT**: Post-LN + Layer Norm
- **GPT**: Pre-LN + Layer Norm
- **LLaMA**: Pre-LN + RMS Norm
- **T5**: Pre-LN + RMS Norm